# LocalFold on this runtime's GPU

Run the cells. The last one prints **one line**: paste it into LocalFold's
model row, where the dropdown says *Colab*.

**What this is.** LocalFold folds in your browser on your own GPU. This
notebook puts the *same code* on the GPU Colab lends you - a real Chrome with
real WebGPU, this repository's own `web/app.js`, driven over CDP. There is no
second implementation to keep in step and no Python re-port of the model: it
is the page you already use, on a bigger card.

**What it is not.** It is not a service. The tunnel lives as long as this
notebook does, the token is yours alone, and it is your runtime being spent.
Colab is for interactive use by the person sitting at it.

Runtime → Change runtime type → **T4 GPU** (or better) before you start.

In [ ]:
#@title 1 · What card did we get?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# 🔴 NO GPU HERE MEANS THE RUNTIME TYPE IS WRONG, and everything below would
# still 'work' - Chrome falls back to SwiftShader, which is a software
# renderer, and a fold takes minutes while looking exactly like success.
# Runtime -> Change runtime type -> T4 GPU.

In [ ]:
#@title 2 · Chrome, Vulkan and the repository (about two minutes)
%%bash
set -e
# 🔴 THE USERSPACE HALF OF THE DRIVER, MATCHED TO THE KERNEL HALF. A runtime
# ships the NVIDIA kernel module and `nvidia-smi`, and NOT the Vulkan ICD that
# Chrome needs - so a Chrome that asks for Vulkan is handed SwiftShader, the
# CPU renderer, and every fold after that is a CPU fold that looks exactly
# like success. Measured on the first real runtime: vendor 'google',
# architecture 'swiftshader', no shader-f16, a 1 GiB buffer ceiling.
#
# The package is versioned by the DRIVER's major number, which differs between
# runtimes, so it is read from nvidia-smi rather than written down.
DRIVER=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
echo "driver major: ${DRIVER:-none}"
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y libvulkan1 vulkan-tools dbus > /dev/null 2>&1
if [ -n "$DRIVER" ]; then
  apt-get -qq install -y "libnvidia-gl-${DRIVER}" > /dev/null 2>&1 \
    || echo "no libnvidia-gl-${DRIVER}; falling back to whatever apt offers"
fi

# Chrome stable, from Google's own repository.
if ! command -v google-chrome > /dev/null; then
  wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null 2>&1
fi
google-chrome --version

# ...and the tunnel, which is how a page on another machine reaches this one.
if ! command -v cloudflared > /dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
fi
cloudflared --version

# The repository. Shallow, and PULLED when it is already here: a runtime that
# has run this before would otherwise keep the clone it made, fixes included.
if [ -d /content/localfold ]; then
  git -C /content/localfold pull -q --ff-only || true
else
  git clone -q --depth 1 https://github.com/sokrypton/localfold /content/localfold
fi
git -C /content/localfold log --oneline -1

# 🔴 THE LINE THAT DECIDES EVERYTHING BELOW. An ICD file here is the driver
# telling Vulkan how to reach the card; `vulkaninfo` naming the GPU is Vulkan
# having done it. Neither of those, and cell 3 will report SwiftShader.
echo "--- Vulkan ICDs:"; ls -1 /usr/share/vulkan/icd.d/ 2>/dev/null || echo "(none)"
echo "--- vulkaninfo:"; vulkaninfo --summary 2>/dev/null \
  | grep -iE 'deviceName|driverName|apiVersion' | head -6 || echo "(vulkaninfo found nothing)"


In [ ]:
#@title 3 · Start the backend and open the door
import json, re, subprocess, sys, time, threading, queue

REPO = '/content/localfold'
PORT = 8710

def reader(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())

# 🔴 THE BACKEND SERVES THE CHECKOUT AND DRIVES CHROME ITSELF. It prints one
# line beginning BACKEND carrying the token it generated and what the adapter
# turned out to be - which is the number worth reading before any fold: an
# adapter named 'SwiftShader' is the CPU, and a fold on it proves nothing.
backend = subprocess.Popen(
    [sys.executable, 'tools/colab_backend.py', '--port', str(PORT)],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = queue.Queue()
threading.Thread(target=reader, args=(backend.stdout, lines), daemon=True).start()

token, adapter = None, None
deadline = time.time() + 300
while time.time() < deadline and token is None:
    try:
        line = lines.get(timeout=5)
    except queue.Empty:
        continue
    print(line)
    if line.startswith('BACKEND '):
        said = json.loads(line[len('BACKEND '):])
        token, adapter = said['token'], said['gpu']

if token is None:
    raise SystemExit('the backend did not come up; the log above says why')
print('\nadapter:', json.dumps(adapter, indent=1))
if not adapter.get('webgpu'):
    print('\n🔴 NO WEBGPU AT ALL - cell 2 did not give Chrome a Vulkan driver.')
elif 'swiftshader' in json.dumps(adapter).lower() or 'llvmpipe' in json.dumps(adapter).lower():
    print('\n🔴 THIS IS THE CPU WEARING A GPU\'S CLOTHES (SwiftShader).'
          ' A fold will run and mean nothing. Check cell 1 has a GPU runtime.')

In [ ]:
#@title 4 · The line to paste into LocalFold
import re, subprocess, time

# 🔴 A QUICK TUNNEL, WHICH IS PUBLIC WHILE IT LIVES. Anyone who learns the URL
# can reach this runtime, which is why the backend refuses every request that
# does not carry the token - and why the line below carries the token in its
# fragment (`#`), the one part of a URL a browser never sends to a server.
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url = None
deadline = time.time() + 120
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if found:
        url = found.group(0)

if url is None:
    raise SystemExit('no tunnel; run this cell again')

handle = f'{url}#{token}'
print('\nPaste this into LocalFold (model row -> Colab):\n')
print('   ' + handle + '\n')

# ...and a check that the door really opens, from outside the runtime's own
# loopback: a tunnel that is up but not yet routing answers 502 for a second
# or two, and 'it printed a URL' is not 'the URL works'.
import urllib.request, json as _json
for attempt in range(10):
    try:
        with urllib.request.urlopen(f'{url}/health?t={token}', timeout=20) as answer:
            print('health:', _json.dumps(_json.loads(answer.read())['gpu'])[:200])
            break
    except Exception as cause:
        print('waiting for the tunnel…', cause)
        time.sleep(3)

### Keep this tab open

The runtime is yours and the tunnel is a process in it: closing this notebook
ends both, and the URL stops working. Colab will also reclaim an idle runtime
on its own schedule.

To check it by hand:

```bash
curl "$URL/health?t=$TOKEN"
curl -X POST "$URL/fold?t=$TOKEN" -H 'content-type: application/json' \
     -d '{"sequence": "GWSTELEKHRSVQ", "model": "af3", "steps": 3, "recycles": 1}'
```